# FluidML Tutorial: HVAC Regression, Iris Classification, and Digits Classification

This notebook is a tutorial for testing the FluidML framework on three complete examples:

1. Multi-target HVAC regression
2. Classification with the iris dataset
3. Classification with the digits dataset

For each example, the notebook will:

- create or save a CSV dataset
- run `fluidml_cli.py quick-start`
- show the generated output directory
- explain how to run HLS
- validate HLS predictions against Python and the truth labels once `Y_hls_pred.csv` is available


## Before You Start

Run this notebook from the `FluidML` repository root.

Typical setup:

```bash
pip install -r requirements.txt
```

If you want to run synthesis after export, make sure either `vivado_hls` or `vitis_hls` is already installed and available on your PATH.


In [54]:
from pathlib import Path
import pickle
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score

repo_root = Path.cwd()
if not (repo_root / "fluidml").exists():
    raise RuntimeError("Run this notebook from the FluidML repository root.")

sys.path.insert(0, str(repo_root))
from fluidml import FluidMLFramework

tutorial_data_dir = repo_root / "tutorial_datasets"
tutorial_data_dir.mkdir(exist_ok=True)

print("Repo root:", repo_root)
print("Dataset folder:", tutorial_data_dir)
print("FluidML import OK")


Repo root: /mnt/c/Users/Mo/Final_improved_framwork/FluidML
Dataset folder: /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets
FluidML import OK


In [55]:
def save_dataframe(df: pd.DataFrame, filename: str) -> Path:
    path = tutorial_data_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved dataset to: {path}")
    return path


def run_cli(args):
    cmd = [sys.executable, "fluidml_cli.py", *args]
    print("Running command:")
    print(" ".join(cmd))
    completed = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(
            "FluidML CLI failed with exit code "
            f"{completed.returncode}\n"
            f"Command: {' '.join(cmd)}"
        )
    return completed


def get_hls_csv(project_dir: str, project_name: str = "fluidml_project") -> Path:
    return repo_root / project_dir / project_name / "solution1" / "csim" / "build" / "Y_hls_pred.csv"


def show_generated_files(project_dir: str):
    project_path = repo_root / project_dir
    if not project_path.exists():
        print(f"Project directory not found: {project_path}")
        return
    print(f"Generated files in {project_path}:")
    for item in sorted(project_path.iterdir()):
        print(" -", item.name)


def show_hls_next_step(project_dir: str, backend: str, project_name: str = "fluidml_project") -> Path:
    tool = "vivado_hls" if backend == "vivado_hls" else "vitis_hls"
    hls_csv = get_hls_csv(project_dir, project_name)
    print("Quick-start generated the HLS project, but it does not run HLS automatically.")
    print("If you are running inside Jupyter, use:")
    print(f"  %cd {project_dir}")
    print(f"  !{tool} -f {project_name}.tcl")
    print("  %cd ..")
    print()
    print("If you are running in a terminal, use:")
    print(f"  cd {project_dir}")
    print(f"  {tool} -f {project_name}.tcl")
    print()
    print("Expected HLS prediction CSV:")
    print(f"  {hls_csv}")
    print()
    if hls_csv.exists():
        print("HLS CSV found. Validation can run now.")
    else:
        print("HLS CSV not found yet. Run HLS first, then rerun this cell.")
    return hls_csv


def validate_regression(project_dir: str, target_names):
    project_path = repo_root / project_dir
    hls_csv = get_hls_csv(project_dir)
    if not hls_csv.exists():
        print(f"HLS CSV not found yet: {hls_csv}")
        print("Run HLS first, then rerun this cell.")
        return None

    if isinstance(target_names, str):
        target_names = [target_names]
    else:
        target_names = list(target_names)

    y_true = np.load(project_path / "Y_test.npy")
    y_pred = np.load(project_path / "Y_pred.npy")

    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)

    scaler_path = project_path / "scaler_y.pkl"
    if scaler_path.exists():
        with open(scaler_path, "rb") as f:
            scaler_y = pickle.load(f)
        y_true = scaler_y.inverse_transform(y_true)
        y_pred = scaler_y.inverse_transform(y_pred)

    y_hls = pd.read_csv(hls_csv).to_numpy(dtype=float)
    if y_hls.ndim == 1:
        y_hls = y_hls.reshape(-1, 1)

    rows = min(len(y_true), len(y_pred), len(y_hls))
    y_true = y_true[:rows]
    y_pred = y_pred[:rows]
    y_hls = y_hls[:rows]

    n_targets = y_true.shape[1]
    if len(target_names) != n_targets:
        target_names = [f"target_{i}" for i in range(n_targets)]

    summary = pd.DataFrame(
        [
            {
                "comparison": "Python vs truth",
                "MAE": mean_absolute_error(y_true, y_pred),
                "R2": r2_score(y_true, y_pred),
            },
            {
                "comparison": "HLS vs truth",
                "MAE": mean_absolute_error(y_true, y_hls),
                "R2": r2_score(y_true, y_hls),
            },
            {
                "comparison": "HLS vs Python",
                "MAE": mean_absolute_error(y_pred, y_hls),
                "R2": r2_score(y_pred, y_hls),
            },
        ]
    )
    print(summary.to_string(index=False))

    preview_data = {}
    for i, name in enumerate(target_names):
        preview_data[f"{name}_truth"] = y_true[:, i]
        preview_data[f"{name}_python"] = y_pred[:, i]
        preview_data[f"{name}_hls"] = y_hls[:, i]
    preview = pd.DataFrame(preview_data)
    return summary, preview.head(10)


def validate_classification(project_dir: str):
    project_path = repo_root / project_dir
    hls_csv = get_hls_csv(project_dir)
    if not hls_csv.exists():
        print(f"HLS CSV not found yet: {hls_csv}")
        print("Run HLS first, then rerun this cell.")
        return None

    y_true = np.load(project_path / "Y_test.npy", allow_pickle=True).reshape(-1)
    y_pred = np.load(project_path / "Y_pred.npy", allow_pickle=True).reshape(-1)
    y_proba = np.load(project_path / "Y_pred_proba.npy")
    hls_probs = pd.read_csv(hls_csv).to_numpy(dtype=float)

    with open(project_path / "fluidml_model.pkl", "rb") as f:
        model = pickle.load(f)
    class_labels = [label.item() if hasattr(label, "item") else label for label in model.classes_]

    hls_pred = np.asarray([class_labels[idx] for idx in np.argmax(hls_probs, axis=1)], dtype=object)
    rows = min(len(y_true), len(y_pred), len(hls_pred), len(hls_probs), len(y_proba))
    y_true = y_true[:rows]
    y_pred = y_pred[:rows]
    hls_pred = hls_pred[:rows]
    y_proba = y_proba[:rows]
    hls_probs = hls_probs[:rows]

    summary = {
        "Python baseline acc": accuracy_score(y_true, y_pred),
        "HLS acc vs truth": accuracy_score(y_true, hls_pred),
        "HLS/Baseline agree": accuracy_score(y_pred, hls_pred),
        "Mean abs prob diff": float(np.mean(np.abs(y_proba - hls_probs))),
        "Max abs prob diff": float(np.max(np.abs(y_proba - hls_probs))),
    }
    for key, value in summary.items():
        print(f"{key}: {value:.6f}" if isinstance(value, float) else f"{key}: {value}")

    cm = confusion_matrix(y_true, hls_pred, labels=class_labels)
    cm_df = pd.DataFrame(cm, index=class_labels, columns=class_labels)
    preview = pd.DataFrame(
        {
            "truth": y_true,
            "python_pred": y_pred,
            "hls_pred": hls_pred,
            "python_confidence": np.max(y_proba, axis=1),
            "hls_confidence": np.max(hls_probs, axis=1),
        }
    )
    return summary, cm_df, preview.head(10)


## 1. HVAC Regression Example

We use the real-world HVAC dataset from your previous framework tests. This is a strong regression example because it predicts three targets at once: `Elec_Cons`, `Therm_Eng_Cons`, and `PMV`.


In [62]:
hvac_url = "https://raw.githubusercontent.com/AbuAli3/ee/main/alldata.csv"
hvac_df = pd.read_csv(hvac_url)
hvac_csv = save_dataframe(hvac_df, "hvac_multi_target.csv")
hvac_df.head()


OSError: Cannot save file into a non-existent directory: '/mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets'

In [61]:
run_cli(
    [
        "quick-start",
        "--data", str(hvac_csv),
        "--features", "Occupancy,Rel_Hum,Room_Temp,Air_Flow_Rat,Air_Temp",
        "--targets", "Elec_Cons,Therm_Eng_Cons,PMV",
        "--backend", "vivado_hls",
        "--precision", "18.8",
        "--output", "tutorial_hvac",
        "--verbose",
    ]
)
show_generated_files("tutorial_hvac")


Running command:
/home/mo/.pyenv/versions/3.10.11/bin/python3.10 fluidml_cli.py quick-start --data /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/hvac_multi_target.csv --features Occupancy,Rel_Hum,Room_Temp,Air_Flow_Rat,Air_Temp --targets Elec_Cons,Therm_Eng_Cons,PMV --backend vivado_hls --precision 18.8 --output tutorial_hvac --verbose

  ███████╗██╗     ██╗   ██╗██╗██████╗     ███╗   ███╗██╗
  ██╔════╝██║     ██║   ██║██║██╔══██╗    ████╗ ████║██║
  █████╗  ██║     ██║   ██║██║██║  ██║    ██╔████╔██║██║
  ██╔══╝  ██║     ██║   ██║██║██║  ██║    ██║╚██╔╝██║██║
  ██║     ███████╗╚██████╔╝██║██████╔╝    ██║ ╚═╝ ██║███████╗
  ╚═╝     ╚══════╝ ╚═════╝ ╚═╝╚═════╝     ╚═╝     ╚═╝╚══════╝
                                                            
                 FluidML by Mohammed Mshragi


2026-06-14 23:31:29 - fluidml.cli - INFO - FluidML Quick Start
2026-06-14 23:31:29 - fluidml.cli - INFO - ========================================
2026-06-14 23:31:29 - fluidml.cli 

RuntimeError: FluidML CLI failed with exit code 1
Command: /home/mo/.pyenv/versions/3.10.11/bin/python3.10 fluidml_cli.py quick-start --data /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/hvac_multi_target.csv --features Occupancy,Rel_Hum,Room_Temp,Air_Flow_Rat,Air_Temp --targets Elec_Cons,Therm_Eng_Cons,PMV --backend vivado_hls --precision 18.8 --output tutorial_hvac --verbose

## Run Vivado HLS


In [38]:
%cd tutorial_hvac
!vivado_hls -f fluidml_project.tcl
%cd ..


/mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_regression

****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2020.1 (64-bit)
  **** SW Build 2902540 on Wed May 27 19:54:35 MDT 2020
  **** IP Build 2902112 on Wed May 27 22:43:36 MDT 2020
    ** Copyright 1986-2020 Xilinx, Inc. All Rights Reserved.

source /tools/Xilinx/Vivado/2020.1/scripts/vivado_hls/hls.tcl -notrace
INFO: Applying HLS Y2K22 patch v1.2 for IP revision
INFO: [HLS 200-10] Running '/tools/Xilinx/Vivado/2020.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'mo' on host 'Moh.localdomain' (Linux_x86_64 version 6.18.33.1-microsoft-standard-WSL2) on Sun Jun 14 20:31:27 BST 2026
INFO: [HLS 200-10] On os Ubuntu 18.04.2 LTS
INFO: [HLS 200-10] In directory '/mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_regression'
Sourcing Tcl script 'fluidml_project.tcl'
INFO: [HLS 200-10] Creating and opening project '/mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_regression/fluid

In [58]:
hvac_hls_csv = show_hls_next_step("tutorial_hvac", "vivado_hls")
print()
if hvac_hls_csv.exists():
    hvac_validation = validate_regression(
        "tutorial_hvac",
        ["Elec_Cons", "Therm_Eng_Cons", "PMV"],
    )
    hvac_validation
else:
    print("Come back to this cell after Vivado HLS finishes csim and creates Y_hls_pred.csv.")


Quick-start generated the HLS project, but it does not run HLS automatically.
If you are running inside Jupyter, use:
  %cd tutorial_hvac
  !vivado_hls -f fluidml_project.tcl
  %cd ..

If you are running in a terminal, use:
  cd tutorial_hvac
  vivado_hls -f fluidml_project.tcl

Expected HLS prediction CSV:
  /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_hvac/fluidml_project/solution1/csim/build/Y_hls_pred.csv

HLS CSV not found yet. Run HLS first, then rerun this cell.

Come back to this cell after Vivado HLS finishes csim and creates Y_hls_pred.csv.


## 2. Iris Classification Example

This example uses string class labels (`setosa`, `versicolor`, `virginica`) so students can see that FluidML handles nonnumeric classification targets.


In [59]:
iris = load_iris(as_frame=True)
iris_df = iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length",
        "sepal width (cm)": "sepal_width",
        "petal length (cm)": "petal_length",
        "petal width (cm)": "petal_width",
        "target": "species_id",
    }
)
iris_df["species"] = iris_df["species_id"].map(dict(enumerate(iris.target_names)))
iris_df = iris_df.drop(columns=["species_id"])
iris_csv = save_dataframe(iris_df, "iris_classification.csv")
iris_df.head()


Saved dataset to: /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/iris_classification.csv


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [60]:
run_cli(
    [
        "quick-start",
        "--data", str(iris_csv),
        "--features", "sepal_length,sepal_width,petal_length,petal_width",
        "--targets", "species",
        "--task", "classification",
        "--backend", "vitis_hls",
        "--precision", "18.8",
        "--output", "tutorial_iris",
        "--verbose",
    ]
)
show_generated_files("tutorial_iris")


Running command:
/home/mo/.pyenv/versions/3.10.11/bin/python3.10 fluidml_cli.py quick-start --data /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/iris_classification.csv --features sepal_length,sepal_width,petal_length,petal_width --targets species --task classification --backend vitis_hls --precision 18.8 --output tutorial_iris --verbose

  ███████╗██╗     ██╗   ██╗██╗██████╗     ███╗   ███╗██╗
  ██╔════╝██║     ██║   ██║██║██╔══██╗    ████╗ ████║██║
  █████╗  ██║     ██║   ██║██║██║  ██║    ██╔████╔██║██║
  ██╔══╝  ██║     ██║   ██║██║██║  ██║    ██║╚██╔╝██║██║
  ██║     ███████╗╚██████╔╝██║██████╔╝    ██║ ╚═╝ ██║███████╗
  ╚═╝     ╚══════╝ ╚═════╝ ╚═╝╚═════╝     ╚═╝     ╚═╝╚══════╝
                                                            
                 FluidML by Mohammed Mshragi


2026-06-14 23:27:52 - fluidml.cli - INFO - FluidML Quick Start
2026-06-14 23:27:52 - fluidml.cli - INFO - ========================================
2026-06-14 23:27:52 - fluidml.cl

In [ ]:
iris_hls_csv = show_hls_next_step("tutorial_iris", "vitis_hls")
print()
if iris_hls_csv.exists():
    iris_validation = validate_classification("tutorial_iris")
    iris_validation
else:
    print("Come back to this cell after Vitis HLS finishes csim and creates Y_hls_pred.csv.")


## 3. Digits Classification Example

The digits dataset has many numeric pixel columns, so this example is useful for testing larger single-target multiclass classification.


In [49]:
digits = load_digits(as_frame=True)
digits_df = digits.frame.rename(columns={"target": "digit"})
digits_csv = save_dataframe(digits_df, "digits_classification.csv")
digits_df.head()


Saved dataset to: /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/digits_classification.csv


,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7,digit
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0,1
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0,2
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0,3
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0,4


In [50]:
run_cli(
    [
        "quick-start",
        "--data", str(digits_csv),
        "--targets", "digit",
        "--task", "classification",
        "--backend", "vivado_hls",
        "--precision", "18.8",
        "--output", "tutorial_digits",
        "--verbose",
    ]
)
show_generated_files("tutorial_digits")


Running command:
/home/mo/.pyenv/versions/3.10.11/bin/python3.10 fluidml_cli.py quick-start --data /mnt/c/Users/Mo/Final_improved_framwork/FluidML/tutorial_datasets/digits_classification.csv --targets digit --task classification --backend vivado_hls --precision 18.8 --output tutorial_digits --verbose

  ███████╗██╗     ██╗   ██╗██╗██████╗     ███╗   ███╗██╗
  ██╔════╝██║     ██║   ██║██║██╔══██╗    ████╗ ████║██║
  █████╗  ██║     ██║   ██║██║██║  ██║    ██╔████╔██║██║
  ██╔══╝  ██║     ██║   ██║██║██║  ██║    ██║╚██╔╝██║██║
  ██║     ███████╗╚██████╔╝██║██████╔╝    ██║ ╚═╝ ██║███████╗
  ╚═╝     ╚══════╝ ╚═════╝ ╚═╝╚═════╝     ╚═╝     ╚═╝╚══════╝
                                                            
                 FluidML by Mohammed Mshragi


2026-06-14 23:07:20 - fluidml.cli - INFO - FluidML Quick Start
2026-06-14 23:07:20 - fluidml.cli - INFO - ========================================
2026-06-14 23:07:20 - fluidml.cli - INFO - Using precision: ap_fixed<18,8>
2026-06-14 23:0

## Suggested Workflow

For each project directory generated by this notebook:

1. Run the FluidML quick-start cell
2. Open the generated folder
3. Run either `vivado_hls -f fluidml_project.tcl` or `vitis_hls -f fluidml_project.tcl`
4. Rerun the validation cell in this notebook
5. Compare:
   - truth vs Python
   - truth vs HLS
   - Python vs HLS

